## Import dependencies

In [1]:
import time
from datetime import datetime
import kubeflow.trainer
import torch


In [2]:
# Verification of dependencies and their version
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")
print(f"PyTorch version: {torch.__version__}")
print("All imports successful!")


Kubeflow version: 0.3.0
PyTorch version: 2.3.1+cu121
All imports successful!


In [3]:
config = kubeflow.trainer.KubernetesBackendConfig()
trainer = kubeflow.trainer.TrainerClient(backend_config=config)

In [4]:
# Set your distributed environment configuration here
NUM_NODES = 4
RESOURCES_PER_NODE = {
    "nvidia.com/gpu": 1,  # GPUs per node
    "cpu": "2",           # CPUs per node
    "memory": "32Gi"       # Memory in GiB per node
}


In [5]:
def get_torch_dist():
    import os
    import torch
    import torch.distributed as dist

    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    dist.init_process_group(backend)
    print("PyTorch Distributed Environment")
    print(f"Using device: {device}")
    print(f"WORLD_SIZE: {dist.get_world_size()}")
    print(f"RANK: {dist.get_rank()}")
    print(f"LOCAL_RANK: {os.environ['LOCAL_RANK']}")
    dist.destroy_process_group()


In [6]:
job_id = trainer.train(
    runtime=trainer.get_runtime("torch-distributed"),
    trainer=kubeflow.trainer.CustomTrainer(
        func=get_torch_dist,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
    ),
)

In [7]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")



Job ID: ra24d7eee10f
Job Status: Created
Creation Time: 2026-01-20 13:21:16+00:00

Job details: TrainJob(name='ra24d7eee10f', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=4, creation_timestamp=datetime.datetime(2026, 1, 20, 13, 21, 16, tzinfo=TzInfo(0)), status='Created')


In [8]:
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=False))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


Waiting for job logs...
[13:21:17] Waiting... (1s)
[13:21:18] Waiting... (2s)
[13:21:19] Waiting... (3s)
[13:21:20] Waiting... (4s)
[13:21:21] Waiting... (5s)
[13:21:22] Waiting... (6s)
Logs received after 6 seconds:
  PyTorch Distributed Environment
  Using device: cuda
  WORLD_SIZE: 4
  RANK: 0
  LOCAL_RANK: 0


In [9]:
for logline in trainer.get_job_logs(job_id, follow=True):
    print(logline)

PyTorch Distributed Environment
Using device: cuda
WORLD_SIZE: 4
RANK: 0
LOCAL_RANK: 0
